# C16 — Post-hoc Logit Adjustment (τ=0.5) on LS ε=0.1

**Tier-2 — без retrain.** Загружаем LS чекпоинт и применяем logit adjustment только на inference.

**Primary base:** `label_smoothing_eps01_2ep` (c03)  
**Fallback:** `label_smoothing_rdrop_eps01_alpha10_2ep` (c12), если c03-веса не сохранены локально

**Baseline (c03):** F1 0.623, worst gap 0.318, flip 0.089

**Outputs**
- Summary CSV: `figures/challengers/c16_logit_adjustment_posthoc_summary.csv`
- Results: `notebooks/results/challenger_training/c16_logit_adjustment_posthoc/`

City-swap eval — в последней ячейке.

In [9]:
import gc
import json
import re

import joblib
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

CWD = Path.cwd()
NOTEBOOKS_DIR = CWD.parent if CWD.name == "challengers" else CWD
PROJECT_ROOT = NOTEBOOKS_DIR.parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures" / "challengers"
RESULTS_DIR = NOTEBOOKS_DIR / "results" / "challenger_training" / "c16_logit_adjustment_posthoc"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR_CANDIDATES = [
    NOTEBOOKS_DIR / "models" / "challengers",
    PROJECT_ROOT / "models" / "challengers",
    PROJECT_ROOT / "notebooks" / "models" / "challengers",
]
BASE_MODEL_CANDIDATES = [
    "label_smoothing_eps01_2ep",
    "label_smoothing_rdrop_eps01_alpha10_2ep",
]

TAU_VALUES = [0.3, 0.5, 0.7]
PRIMARY_TAU = 0.5
SUMMARY_CSV = FIGURES_DIR / "c16_logit_adjustment_posthoc_summary.csv"
BATCH_SIZE = 8
MAX_LENGTH = 128


def resolve_model_dir(model_name):
    for model_root in MODEL_DIR_CANDIDATES:
        model_dir = model_root / model_name
        if (model_dir / "config.json").exists():
            return model_dir
    return None


def list_available_models():
    found = []
    seen = set()
    for model_root in MODEL_DIR_CANDIDATES:
        if not model_root.exists():
            continue
        for path in sorted(model_root.iterdir()):
            if path.is_dir() and (path / "config.json").exists() and path.name not in seen:
                seen.add(path.name)
                found.append(str(path))
    return found


BASE_MODEL_DIR = None
BASE_MODEL_NAME = None
for candidate in BASE_MODEL_CANDIDATES:
    resolved = resolve_model_dir(candidate)
    if resolved is not None:
        BASE_MODEL_DIR = resolved
        BASE_MODEL_NAME = candidate
        break

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"BASE_MODEL_NAME: {BASE_MODEL_NAME}")
print(f"BASE_MODEL_DIR: {BASE_MODEL_DIR}")
print(f"device: {device}")

if BASE_MODEL_DIR is None:
    available = list_available_models()
    raise FileNotFoundError(
        "No base model found. Checked: "
        + ", ".join(BASE_MODEL_CANDIDATES)
        + "\nAvailable models:\n  - "
        + "\n  - ".join(available or ["(none)"])
        + "\nRun c03_label_smoothing_tuning.ipynb to train label_smoothing_eps01_2ep."
    )

if BASE_MODEL_NAME != BASE_MODEL_CANDIDATES[0]:
    print(f"WARNING: using fallback base model {BASE_MODEL_NAME}")

BASE_MODEL_NAME: label_smoothing_eps01_2ep
BASE_MODEL_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/challengers/label_smoothing_eps01_2ep
device: cpu


In [10]:
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_test = pd.read_csv(DATA_DIR / "test.csv")

mapping = pd.read_csv(DATA_DIR / "label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))
for df in [df_train, df_test]:
    df["supercategory"] = df["label"].map(label_to_supercat)

le_path = BASE_MODEL_DIR / "label_encoder.joblib"
if le_path.exists():
    le = joblib.load(le_path)
else:
    le = LabelEncoder()
    le.fit(df_train["supercategory"])
    print("label_encoder.joblib not found — fitted LabelEncoder from train.csv")

df_train["y"] = le.transform(df_train["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])
num_labels = len(le.classes_)

class_counts = df_train["y"].value_counts().sort_index().to_numpy()
class_priors = class_counts / class_counts.sum()
log_priors = torch.tensor(np.log(np.clip(class_priors, 1e-12, None)), dtype=torch.float32)

print(f"Test: {len(df_test)}, labels: {num_labels}")
print(f"Class priors: {class_priors.round(4).tolist()}")

Test: 5510, labels: 9
Class priors: [0.0935, 0.1762, 0.1288, 0.1214, 0.0623, 0.2001, 0.0615, 0.1199, 0.0364]


In [11]:
tokenizer = AutoTokenizer.from_pretrained(str(BASE_MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(BASE_MODEL_DIR))
model = model.to(device).eval()


def predict_logits(texts, batch_size=BATCH_SIZE):
    all_logits = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        all_logits.append(logits.cpu())
        del enc, logits
    return torch.cat(all_logits, dim=0)


def adjusted_preds(logits, tau):
    adjusted = logits + tau * log_priors
    return adjusted.argmax(dim=-1).numpy()


def ovr_rates(df, group_col, num_classes):
    groups = sorted(df[group_col].dropna().unique())
    tpr = np.zeros((len(groups), num_classes))
    support = np.zeros((len(groups), num_classes))
    for gi, group_name in enumerate(groups):
        dg = df[df[group_col] == group_name]
        yt, yp = dg["y_true"].values, dg["y_pred"].values
        for c in range(num_classes):
            positive_mask = yt == c
            tp = np.sum((yp == c) & positive_mask)
            fn = np.sum((yp != c) & positive_mask)
            support[gi, c] = positive_mask.sum()
            tpr[gi, c] = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    return tpr, support


def robust_gaps(tpr, support, min_support=30):
    gaps = []
    for c in range(tpr.shape[1]):
        col = tpr[support[:, c] >= min_support, c]
        col = col[~np.isnan(col)]
        gaps.append(col.max() - col.min() if len(col) >= 2 else np.nan)
    gaps = np.array(gaps)
    valid = gaps[~np.isnan(gaps)]
    return (valid.max() if len(valid) else np.nan, valid.mean() if len(valid) else np.nan)


test_texts = df_test["resume_text"].fillna("").astype(str).tolist()
print("Computing base logits...")
base_logits = predict_logits(test_texts)
y_true = df_test["y"].values

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1652.42it/s, Materializing param=classifier.weight]                                      


Computing base logits...


In [12]:
summary_rows = []

for tau in TAU_VALUES:
    y_pred = adjusted_preds(base_logits, tau)
    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro"))

    df_eval = df_test.copy()
    df_eval["y_true"] = y_true
    df_eval["y_pred"] = y_pred
    tpr, support = ovr_rates(df_eval, "city_group", num_labels)
    worst_gap, macro_gap = robust_gaps(tpr, support, min_support=30)

    row = {
        "base_model": BASE_MODEL_NAME,
        "method": "Post-hoc Logit Adjustment on LS eps01",
        "tau": tau,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "worst_gap": float(worst_gap),
        "macro_gap": float(macro_gap),
    }
    summary_rows.append(row)
    print(f"tau={tau:.1f}: Acc={acc:.4f}, F1={macro_f1:.4f}, worst_gap={worst_gap:.4f}, macro_gap={macro_gap:.4f}")

summary_df = pd.DataFrame(summary_rows).sort_values(["worst_gap", "macro_f1"], ascending=[True, False])
summary_df.to_csv(SUMMARY_CSV, index=False)
summary_df.to_csv(RESULTS_DIR / "posthoc_summary.csv", index=False)

best_tau = float(summary_df.iloc[0]["tau"])
best_pred = adjusted_preds(base_logits, best_tau)
pred_df = df_test[["city_group", "label", "supercategory"]].copy()
pred_df["y_true"] = y_true
pred_df["y_pred"] = best_pred
pred_df.to_csv(RESULTS_DIR / "predictions_test_best_tau.csv", index=False)

config = {
    "method": "Post-hoc Logit Adjustment on LS eps01",
    "base_model": BASE_MODEL_NAME,
    "tau_values": TAU_VALUES,
    "best_tau": best_tau,
    "results": summary_rows,
}
with open(RESULTS_DIR / "posthoc_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"\nBest tau by gap: {best_tau}")
print(f"Saved: {SUMMARY_CSV}")
summary_df

tau=0.3: Acc=0.6096, F1=0.6260, worst_gap=0.3412, macro_gap=0.1296
tau=0.5: Acc=0.6062, F1=0.6237, worst_gap=0.3471, macro_gap=0.1255
tau=0.7: Acc=0.6056, F1=0.6243, worst_gap=0.3471, macro_gap=0.1250

Best tau by gap: 0.3
Saved: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/challengers/c16_logit_adjustment_posthoc_summary.csv


,base_model,method,tau,accuracy,macro_f1,worst_gap,macro_gap
0,label_smoothing_eps01_2ep,Post-hoc Logit Adjustment on LS eps01,0.3,0.609619,0.626030,0.341176,0.129633
2,label_smoothing_eps01_2ep,Post-hoc Logit Adjustment on LS eps01,0.7,0.605626,0.624311,0.347059,0.125046
1,label_smoothing_eps01_2ep,Post-hoc Logit Adjustment on LS eps01,0.5,0.606171,0.623720,0.347059,0.125457


## City-swap eval (τ = primary / best)

Шаблон для будущих ноутбуков: отдельная ячейка после training/eval.

In [13]:
SWAP_CITIES = ["Москва", "Екатеринбург", "Новосибирск", "Краснодар", "Воронеж"]
CITY_SWAP_PATTERNS = [
    "санкт-петербург", "нижний новгород", "ростов-на-дону", "новосибирск",
    "екатеринбург", "красноярск", "волгоград", "калининград", "владивосток",
    "хабаровск", "ставрополь", "саратов", "челябинск", "самара", "казань",
    "москва", "омск", "воронеж", "пермь", "тюмень", "томск", "уфа",
    "краснодар", "мск", "спб", "питер",
]
CITY_SWAP_RE = re.compile(r"\b(" + "|".join(re.escape(c) for c in CITY_SWAP_PATTERNS) + r")\b", re.IGNORECASE)


def swap_cities_in_text(text, target_city):
    if pd.isna(text):
        return ""
    def replacer(match):
        orig = match.group(0)
        return target_city.capitalize() if orig and orig[0].isupper() else target_city.lower()
    return CITY_SWAP_RE.sub(replacer, str(text))


def predict_adjusted_batch(texts, tau, batch_size=BATCH_SIZE):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
            adjusted = logits + tau * log_priors.to(logits.device)
            preds = adjusted.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        del enc, logits, adjusted, preds
    return np.array(all_preds)


city_swap_tau = PRIMARY_TAU
print(f"City-swap eval: base={BASE_MODEL_NAME}, post-hoc tau={city_swap_tau}")

base_texts = df_test["resume_text"].fillna("").astype(str).tolist()
orig_preds = predict_adjusted_batch(base_texts, city_swap_tau)

any_flip = np.zeros(len(df_test), dtype=bool)
per_city = {}
for swap_city in SWAP_CITIES:
    swapped = df_test["resume_text"].apply(lambda x: swap_cities_in_text(x, swap_city)).fillna("").astype(str).tolist()
    swap_preds = predict_adjusted_batch(swapped, city_swap_tau)
    flipped = swap_preds != orig_preds
    flip_rate = float(flipped.mean())
    any_flip |= flipped
    per_city[swap_city] = flip_rate
    print(f"  {swap_city}: {flip_rate:.3f} ({int(flipped.sum())}/{len(df_test)})")
    del swapped, swap_preds, flipped
    gc.collect()

overall_flip = float(any_flip.mean())
acc = float(accuracy_score(y_true, orig_preds))
f1 = float(f1_score(y_true, orig_preds, average="macro"))
print(f"  Acc={acc:.3f}  F1={f1:.3f}  Flip={overall_flip:.3f}")

city_swap_row = {
    "base_model": BASE_MODEL_NAME,
    "method": "Post-hoc Logit Adjustment on LS eps01",
    "tau": city_swap_tau,
    "accuracy": acc,
    "macro_f1": f1,
    "overall_flip_rate": overall_flip,
}
for city, rate in per_city.items():
    city_swap_row[f"flip_{city}"] = rate

city_swap_df = pd.DataFrame([city_swap_row])
city_swap_df.to_csv(RESULTS_DIR / "city_swap_summary.csv", index=False)
city_swap_df.to_csv(FIGURES_DIR / "c16_logit_adjustment_posthoc_city_swap.csv", index=False)

with open(RESULTS_DIR / "city_swap_summary.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "base_model": BASE_MODEL_NAME,
            "tau": city_swap_tau,
            "accuracy": acc,
            "macro_f1": f1,
            "overall_flip_rate": overall_flip,
            "per_swap_city": {city: {"flip_rate": rate} for city, rate in per_city.items()},
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

city_swap_df

City-swap eval: base=label_smoothing_eps01_2ep, post-hoc tau=0.5
  Москва: 0.016 (87/5510)
  Екатеринбург: 0.033 (182/5510)
  Новосибирск: 0.028 (155/5510)
  Краснодар: 0.039 (213/5510)
  Воронеж: 0.030 (167/5510)
  Acc=0.606  F1=0.624  Flip=0.070


,base_model,method,tau,accuracy,macro_f1,overall_flip_rate,flip_Москва,flip_Екатеринбург,flip_Новосибирск,flip_Краснодар,flip_Воронеж
0,label_smoothing_eps01_2ep,Post-hoc Logit Adjustment on LS eps01,0.5,0.606171,0.62372,0.070236,0.015789,0.033031,0.028131,0.038657,0.030309
